# Exploracion de la simulacion base del ABM de dependencia

Este notebook revisa las salidas de una simulacion base exploratoria del circuito administrativo simplificado del SAAD. Los resultados permiten comprobar que el ABM funciona y genera trayectorias agregadas coherentes, pero no constituyen todavia una validacion empirica final.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from analysis.metrics import calculate_simulation_metrics, metrics_to_dataframe

csv_path = PROJECT_ROOT / "data" / "simulation_outputs" / "base_simulation.csv"
figures_dir = PROJECT_ROOT / "outputs" / "figures"
metrics_path = PROJECT_ROOT / "outputs" / "metrics" / "base_simulation_metrics.json"

## Carga de datos

Se carga el CSV mensual generado por `python src/run_simulation.py`.

In [ ]:
df = pd.read_csv(csv_path)
df.head()

In [ ]:
df.columns.tolist()

In [ ]:
df.describe().T

## Comprobacion de poblacion

La poblacion vulnerable inicial es unica y coincide con los agentes no solicitantes en el mes 0. La suma de estados administrativos debe mantenerse aproximadamente constante e igual al numero de agentes vulnerables simulados.

In [ ]:
estado_cols = [
    "no_solicitantes",
    "pendiente_grado",
    "sin_grado",
    "con_derecho",
    "con_pia",
    "prestacion_efectiva",
    "lista_espera",
]

df["total_estados"] = df[estado_cols].sum(axis=1)
df[["month", "vulnerables", "total_estados"]].head(), df[["month", "vulnerables", "total_estados"]].tail()

In [ ]:
df["diferencia_total"] = df["vulnerables"] - df["total_estados"]
df["diferencia_total"].describe()

## Comprobaciones de coherencia interna

Estas comprobaciones no constituyen una validación empírica del modelo, sino una verificación funcional de la coherencia interna de la simulación. Su finalidad es confirmar que el flujo administrativo implementado produce salidas consistentes antes de iniciar fases posteriores de calibración, generación de escenarios y comparación con indicadores externos como el IMCV.

In [ ]:
expected_columns = [
    "month",
    "vulnerables",
    "no_solicitantes",
    "pendiente_grado",
    "sin_grado",
    "con_derecho",
    "con_pia",
    "prestacion_efectiva",
    "lista_espera",
    "grado_I",
    "grado_II",
    "grado_III",
    "teleasistencia",
    "ayuda_domicilio",
    "atencion_residencial",
    "cuidados_familiares",
]

checks = {
    "61 filas simuladas": len(df) == 61,
    "mes inicial 0": df["month"].min() == 0,
    "mes final 60": df["month"].max() == 60,
    "meses consecutivos 0-60": df["month"].tolist() == list(range(61)),
    "columnas esperadas presentes": set(expected_columns).issubset(df.columns),
    "sin valores negativos": (df[expected_columns] < 0).sum().sum() == 0,
    "poblacion vulnerable constante": df["vulnerables"].nunique() == 1,
    "poblacion vulnerable inicial 6387": df.loc[df["month"] == 0, "vulnerables"].iloc[0] == 6387,
    "no_solicitantes disminuye progresivamente": df["no_solicitantes"].is_monotonic_decreasing,
    "prestacion_efectiva aumenta acumulativamente": df["prestacion_efectiva"].is_monotonic_increasing,
    "lista_espera aumenta acumulativamente": df["lista_espera"].is_monotonic_increasing,
}

pd.Series(checks, name="resultado")

In [ ]:
missing_columns = [column for column in expected_columns if column not in df.columns]
missing_columns

In [ ]:
df.loc[df["month"] == 60, expected_columns].T.rename(columns={df.index[df["month"] == 60][0]: "mes_60"})

## Resultados finales de la simulación base

La siguiente tabla resume los valores agregados del mes 60 para estados administrativos, grados reconocidos y prestaciones asignadas.

In [ ]:
final_result_columns = [
    "no_solicitantes",
    "pendiente_grado",
    "sin_grado",
    "con_derecho",
    "con_pia",
    "prestacion_efectiva",
    "lista_espera",
    "grado_I",
    "grado_II",
    "grado_III",
    "teleasistencia",
    "ayuda_domicilio",
    "atencion_residencial",
    "cuidados_familiares",
]

df.loc[df["month"] == 60, final_result_columns].T.rename(columns={df.index[df["month"] == 60][0]: "mes_60"})

Los resultados obtenidos muestran una evolución progresiva de los agentes a través del flujo administrativo simulado. La disminución de los no solicitantes refleja la entrada gradual de población vulnerable en el sistema, mientras que el aumento de las prestaciones efectivas y de la lista de espera representa la acumulación de agentes en los estados finales del procedimiento. La distribución por grados y por tipos de prestación responde a la parametrización inicial del modelo y deberá ser ajustada en fases posteriores de calibración.

## Métricas internas de la simulación

Las métricas calculadas en esta sección permiten resumir el comportamiento interno de la simulación base. Estas métricas describen la distribución final de los agentes, las tasas acumuladas sobre la población vulnerable inicial, la composición de grados y prestaciones, y algunas comprobaciones básicas de coherencia temporal. No deben interpretarse todavía como métricas de validación empírica, ya que la comparación con indicadores externos como el IMCV se abordará en una fase posterior.

In [ ]:
df_metrics = pd.read_csv(csv_path)
metrics = calculate_simulation_metrics(df_metrics)
metrics_table = metrics_to_dataframe(metrics)
metrics_table

In [ ]:
if metrics_path.exists():
    with metrics_path.open("r", encoding="utf-8") as file:
        saved_metrics = json.load(file)
    saved_metrics_table = metrics_to_dataframe(saved_metrics)
else:
    saved_metrics_table = pd.DataFrame(columns=["metric", "value"])

saved_metrics_table

## Figuras principales

Las figuras resumen la evolucion de estados SAAD, grados de dependencia y prestaciones finales. Deben interpretarse como exploracion funcional del modelo base.

In [ ]:
for figure_name in [
    "evolucion_estados_saad.png",
    "evolucion_grados_dependencia.png",
    "evolucion_prestaciones.png",
]:
    display(Image(filename=str(figures_dir / figure_name)))